# Running Stata in a Jupyter notebook on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/larsvilhuber/jupyter-stata-colab/blob/main/stata_colab_example.ipynb)

This notebook installs Stata into the Colab runtime, asks for **your** Stata license (as a base64-encoded string), and then runs Stata code through the `%%stata` cell magic provided by StataCorp's [PyStata](https://www.stata.com/python/pystata19/).

**Before you start**, create the base64-encoded version of your `stata.lic` on your own computer:

```bash
base64 -w0 /usr/local/stata/stata.lic      # Linux
base64 -i /Applications/Stata/stata.lic    # macOS
```

```powershell
[Convert]::ToBase64String([IO.File]::ReadAllBytes("C:\Program Files\Stata19\stata.lic"))   # Windows
```

Copy the resulting single line; you will paste it into a hidden prompt below. It is only written to the (temporary) Colab virtual machine, never to the notebook.

## 1. Settings

Stata is copied out of one of the [AEA Data Editor's Stata Docker images](https://hub.docker.com/u/dataeditors) (which contain no license). Set the version using Docker repository spelling (`19_5` for Stata 19.5), and choose a published tag or `latest` to look up the most recently updated tag on Docker Hub. Pick the edition that matches your license: an MP license can also run SE, so `se` works for both SE and MP licenses.

In [ ]:
VERSION = "19_5"       # Docker repository spelling, e.g. "19_5" for Stata 19.5
TAG = "2026-08-12"     # a published tag, or "latest"
EDITION = "se"         # "be", "se" or "mp"
STATA_DIR = "/usr/local/stata"

## 2. Install Stata

This downloads the helper script from this repository and copies Stata to `/usr/local/stata` (about 700 MB, usually well under a minute on Colab).

In [ ]:
import os

if not os.path.exists("setup_stata.py"):
    !wget -q https://raw.githubusercontent.com/larsvilhuber/jupyter-stata-colab/main/setup_stata.py

import setup_stata

setup_stata.install_stata(version=VERSION, tag=TAG, edition=EDITION, stata_dir=STATA_DIR)

## 3. Enter your license

Paste the base64-encoded content of your `stata.lic` when prompted (the input is hidden). If the environment variable `STATA_LIC_BASE64` is set, it is used instead of prompting.

In [ ]:
from getpass import getpass

license_b64 = os.environ.get("STATA_LIC_BASE64") or getpass("Base64-encoded stata.lic: ")
setup_stata.write_license(license_b64, stata_dir=STATA_DIR)
del license_b64

## 4. Start Stata

This first checks that Stata accepts the license, then initializes PyStata, which registers the `%stata` and `%%stata` magics.

In [ ]:
setup_stata.configure_pystata(stata_dir=STATA_DIR, edition=EDITION)

## 5. Use Stata

Any cell starting with `%%stata` is run by Stata.

In [ ]:
%%stata
sysuse auto, clear
summarize price mpg weight

In [ ]:
%%stata
regress price mpg weight foreign

Graphs are displayed inline:

In [ ]:
%%stata
twoway (scatter price mpg) (lfit price mpg), title("Price vs. mileage")

Data can be moved between Stata and Python (pandas):

In [ ]:
from pystata import stata

df = stata.pdataframe_from_data()
df.groupby("foreign")[["price", "mpg"]].mean()

In [ ]:
%%stata -d df -force
describe, short